# CodeQA — Seq2seq LSTM + Attention + Copy (paper-aligned)

Notebook baru untuk baseline **Seq2seq**, bukan Mini V3 atau CodeBERT. Model, loss copy, dynamic vocabulary, dan beam search memakai implementasi **NeuralCodeSum/C2NL** yang disertakan di `references/NeuralCodeSum/` (MIT). Repo CodeQA menautkan NeuralCodeSum sebagai rujukan evaluasi; ini **rekonstruksi baseline dari paper + source terkait**, bukan source Seq2seq khusus CodeQA yang dirilis penulis.

**Sumber:** [CodeQA §6.1 dan Appendix B](https://aclanthology.org/2021.findings-emnlp.223.pdf), [CodeQA repo](https://github.com/jadecxliu/CodeQA), [NeuralCodeSum script Python RNN](https://github.com/wasiahmad/NeuralCodeSum/blob/ffc43a415718d80aba4fe8372a438e0e492ddc6d/scripts/python/rnn.sh).

| Komponen | Notebook | Dasar |
|---|---|---|
| RNN | LSTM; encoder bidirectional, decoder unidirectional | Default source rujukan; jenis sel tidak dirinci CodeQA |
| Layer / hidden | 2 / 512 (encoder 256 per arah) | Appendix B + interpretasi source |
| Attention / copy | Luong general, reuse attention, dynamic dictionary | §6.1 + source rujukan |
| Embedding | 512; target embedding diikat ke generator | Script rujukan, tidak dirinci CodeQA |
| Batch / Adam / dropout | 32 / LR 0.002 / 0.2 | Appendix B |
| LR decay / gradient clip | 0.99 per epoch / 5.0 | Script rujukan |
| Beam | 4; block repeated trigram; replace UNK | Beam paper; detail lainnya dari script rujukan |
| Source / target | 400 / 30 token isi (+ BOS/EOS target) | Script rujukan, bukan batas yang dinyatakan untuk Seq2seq di CodeQA |
| Vocab maksimum | Source 50k; target 30k, dibangun **train saja** | Script rujukan + pencegahan kebocoran dev/test |
| Input QA | `[CLS] question [SEP] code`, lowercase, whitespace split | §6.1; penyesuaian source summarization menjadi QA |
| Checkpoint | BLEU dev terbaik; dev greedy, test beam 4 | Pola train/test source rujukan; pilihan dev decoding tidak dirinci CodeQA |

**Mode awal `smoke`:** arsitektur dan vocabulary penuh, hanya data/epoch dibatasi untuk memeriksa pipeline. Ini bukan skor reproduksi penuh.
**Mode `medium`:** 10.000 train, 500 dev, 500 test; maksimum 20 epoch dan patience 4. Arsitektur tetap 2 layer, hidden 512 dengan copy dan batch 32. Batas dua jam per sesi, dengan resume dari checkpoint. Ini eksperimen skala kecil, bukan reproduksi penuh.

**Mode `paper` (default):** seluruh 56.085 train, 7.000 dev/test, maksimum 200 epoch, patience 20. Batas 2 jam per eksekusi dapat menghentikan run sebelum konvergen; resume melanjutkan batch/epoch. Hasil tetap berstatus parsial jika terhenti.

Kernel lokal yang sudah diuji: **CodeQA Seq2seq (PyTorch)**. Di komputer lain, gunakan kernel dengan `torch`, `numpy`, `nltk`, `tqdm`, dan `nbformat`. Tidak perlu Transformers, model pretrained, atau unduhan bobot. Contoh jawaban tidak menjadi input saat generasi. Dataset/checkpoint lama tidak diubah.

In [ ]:
import sys, os, json, time, math, random, hashlib, gc, argparse, importlib.util
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, Markdown

# Pilihan: "smoke" (tes cepat), "medium" (eksperimen), "paper" (data penuh).
PROFILE = "paper"
SEED = 42
MAX_SESSION_SECONDS = 2 * 60 * 60
BATCH_SIZE = 32  # Appendix B; jika diubah, perbedaannya tercatat pada konfigurasi run.
EVAL_BATCH_SIZE = 8
RESUME = True
EVALUATE_PARTIAL = False  # Hindari membuka test untuk memilih konfigurasi run yang belum selesai.
RUN_TAG = "v1"  # Ganti jika ingin eksperimen baru; jangan campur hasil run.

ROOT = Path.cwd()
assert (ROOT / "data/python/train/train.code").is_file(), "Direktori dataset CodeQA tidak ditemukan"
REF = ROOT / "references/NeuralCodeSum"
assert (REF / "UPSTREAM.json").exists(), "Source rujukan yang disertakan notebook belum tersedia"
sys.path.insert(0, str(REF))
from c2nl.config import add_model_args
from c2nl.inputters.vocabulary import Vocabulary
from c2nl.inputters import constants
from c2nl.objects.code import Code
from c2nl.objects.summary import Summary
from c2nl.inputters.vector import vectorize, batchify
from c2nl.utils.copy_utils import make_src_map, collapse_copy_scores
from c2nl.modules.copy_generator import CopyGenerator, CopyGeneratorCriterion
from c2nl.translator import Translator, TranslationBuilder, GNMTGlobalScorer
# Load wrapper dengan nama unik agar tidak berbenturan dengan modul model notebook lain.
spec = importlib.util.spec_from_file_location("codeqa_reference_model", REF / "main/model.py")
wrapper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(wrapper)
Code2NaturalLanguage = wrapper.Code2NaturalLanguage

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(4, os.cpu_count() or 1))
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""): h.update(part)
    return h.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")
    temporary.replace(path)

seed_all(SEED)
print("Python:", sys.version.split()[0], "Torch:", torch.__version__, "Device:", DEVICE)
if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_name(0))
upstream = json.loads((REF / "UPSTREAM.json").read_text())
for relative, expected in upstream["vendored_sha256"].items():
    assert sha256(REF / relative) == expected, f"Source berubah: {relative}"
print("Upstream commit:", upstream["commit"])
print("Compatibility patches:", upstream["compatibility_patches"])

## 1. Data resmi, vocabulary train-only, dan batas representasi

Tokenisasi mengikuti loader C2NL: lowercase dan whitespace split. Tidak memakai BPE Mini V3. Source diawali `[CLS] question [SEP]`, lalu kode. Copy mengambil token dari **seluruh input terlihat**, termasuk pertanyaan. Panjang target 30 di source rujukan berarti 30 token isi, kemudian BOS/EOS ditambahkan. Referensi dev/test disimpan utuh untuk evaluasi.

Data test dibaca saat evaluasi akhir saja. Ukuran dan hash ditulis supaya resume tidak mencampur dataset berbeda.

In [ ]:
assert PROFILE in ("smoke", "medium", "paper")
SIZES = {"train": 56085, "dev": 7000, "test": 7000}
DATA_HASHES = {}
def load_rows(split):
    columns = {}
    for extension in ("code", "question", "answer"):
        path = ROOT / f"data/python/{split}/{split}.{extension}"
        columns[extension] = path.read_text(encoding="utf-8").splitlines()
        DATA_HASHES[f"{split}.{extension}"] = sha256(path)
    assert all(len(x) == SIZES[split] for x in columns.values()), f"Alignment {split} gagal"
    return [{"id": i, **{k: v[i].strip() for k, v in columns.items()}} for i in range(SIZES[split])]

def source_tokens(row):
    return (["[CLS]"] + row["question"].split() + ["[SEP]"] + row["code"].split())

def tokens_uncased(tokens):
    return [token.lower() for token in tokens]

full_train, full_dev = load_rows("train"), load_rows("dev")
source_counts, target_counts = Counter(), Counter()
for row in full_train:
    source_counts.update(tokens_uncased(source_tokens(row))[:400])
    target_counts.update(row["answer"].lower().split()[:30])

def make_vocab(counts, limit):
    vocabulary = Vocabulary()
    for token, _ in counts.most_common():
        if len(vocabulary) >= limit: break
        vocabulary.add(token)
    return vocabulary

src_dict = make_vocab(source_counts, 50000)
tgt_dict = make_vocab(target_counts, 30000)

def choose(rows, n, seed):
    indices = sorted(random.Random(seed).sample(range(len(rows)), min(n, len(rows))))
    return [rows[i] for i in indices]

train_count = {"smoke": 128, "medium": 10000, "paper": len(full_train)}[PROFILE]
dev_count = {"smoke": 32, "medium": 500, "paper": len(full_dev)}[PROFILE]
train_rows = full_train if PROFILE == "paper" else choose(full_train, train_count, SEED)
dev_rows = full_dev if PROFILE == "paper" else choose(full_dev, dev_count, SEED + 1)
parser = argparse.ArgumentParser()
add_model_args(parser)
args = parser.parse_args([])
# Defaults come from pinned C2NL config; overrides from its Python RNN script / CodeQA Appendix B.
vars(args).update(dict(model_type="rnn", rnn_type="LSTM", emsize=512, nhid=512,
    nlayers=2, bidirection=True, use_all_enc_layers=False, use_src_word=True,
    use_tgt_word=True, use_src_char=False, use_tgt_char=False, use_code_type=False,
    max_src_len=400, max_tgt_len=30, src_vocab_size=len(src_dict), tgt_vocab_size=len(tgt_dict),
    copy_attn=True, reuse_copy_attn=True, attn_type="general", share_decoder_embeddings=True,
    conditional_decoding=False, coverage_attn=False, split_decoder=False,
    dropout_rnn=0.2, dropout_emb=0.2, dropout=0.2, optimizer="adam", learning_rate=0.002,
    grad_clipping=5.0, lr_decay=0.99, fix_embeddings=False, uncase=True,
    batch_size=BATCH_SIZE, dataset_weights={1: 1.0}))
MAX_EPOCHS, PATIENCE = {"smoke": (2, 2), "medium": (20, 4), "paper": (200, 20)}[PROFILE]
notebook_source = json.loads((ROOT / "CodeQA_Seq2Seq_Paper.ipynb").read_text())
code_sources = ["".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    for cell in notebook_source["cells"] if cell["cell_type"] == "code"]
adapter_sha256 = hashlib.sha256("\n".join(code_sources).encode()).hexdigest()
config = dict(adapter_sha256=adapter_sha256, profile=PROFILE, seed=SEED, model=vars(args), max_epochs=MAX_EPOCHS,
    patience=PATIENCE, beam=4, validation_decoding="greedy", test_decoding="beam4",
    block_ngram_repeat=3, replace_unk=True, train_ids=[r["id"] for r in train_rows],
    dev_ids=[r["id"] for r in dev_rows], data_hashes=dict(DATA_HASHES),
    upstream_commit=upstream["commit"], source_hashes=upstream["vendored_sha256"], torch_version=torch.__version__)
run_hash = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
OUT = ROOT / "seq2seq_paper_results" / f"{PROFILE}_{RUN_TAG}_{run_hash}"
OUT.mkdir(parents=True, exist_ok=True)
(OUT.parent / ".gitignore").write_text("*.pt\n*.tmp\n")
if (OUT / "config.json").exists():
    assert json.loads((OUT / "config.json").read_text()) == json.loads(json.dumps(config)), "Run config tidak cocok"
write_json(OUT / "config.json", config)
write_json(OUT / "vocab.json", {"source": src_dict.tok2ind, "target": tgt_dict.tok2ind})

def representation_stats(rows):
    answer_tokens = [t for r in rows for t in r["answer"].lower().split()]
    return {"n": len(rows), "source_truncated": sum(len(source_tokens(r)) > 400 for r in rows),
        "target_truncated_for_training": sum(len(r["answer"].split()) > 30 for r in rows),
        "answer_token_oov_pct": 100 * sum(t not in tgt_dict for t in answer_tokens) / max(1, len(answer_tokens))}

print("Source/target vocabulary:", len(src_dict), len(tgt_dict))
print("Train:", representation_stats(train_rows), "Dev:", representation_stats(dev_rows))
print("Run:", OUT)

## 2. Adapter QA ke data C2NL dan generasi tanpa gold answer

Adapter mempertahankan implementasi model/loss upstream. Saat inference, `Summary=None`: jawaban referensi tidak masuk ke decoder. Decoder memakai BOS dan prediksinya sendiri. Dynamic dictionary memungkinkan menyalin kata yang tidak ada pada target vocabulary global.

In [ ]:
class QADataset(Dataset):
    def __init__(self, rows, model, training=False):
        self.rows, self.model, self.training = rows, model, training
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        code = Code(row["id"])
        code.language = 1
        code.tokens = tokens_uncased(source_tokens(row))[:args.max_src_len]
        code.text = " ".join(code.tokens)
        summary = None
        if self.training:
            summary = Summary(row["id"])
            summary.tokens = row["answer"].lower().split()[:args.max_tgt_len]
            assert summary.tokens, f"Jawaban train kosong: {row['id']}"
            summary.text = " ".join(summary.tokens)
            summary.prepend_token(); summary.append_token()
        return vectorize({"code": code, "summary": summary}, self.model)

def make_model():
    seed_all(SEED)
    model = Code2NaturalLanguage(argparse.Namespace(**vars(args)), src_dict, tgt_dict)
    if DEVICE == "cuda": model.cuda()
    model.init_optimizer()
    return model

def loader(rows, model, training=False, epoch=0, start_batch=0):
    dataset = QADataset(rows, model, training)
    if training:
        order = torch.randperm(len(rows), generator=torch.Generator().manual_seed(SEED + epoch)).tolist()
        batches = [order[i:i+BATCH_SIZE] for i in range(0, len(order), BATCH_SIZE)]
        return DataLoader(dataset, batch_sampler=batches[start_batch:], collate_fn=batchify, num_workers=0,
            generator=torch.Generator().manual_seed(SEED + epoch))
    return DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=batchify, num_workers=0,
            generator=torch.Generator().manual_seed(SEED + epoch))

def beam_inputs(batch, model):
    inputs = {k: batch[k] for k in ("code_word_rep", "code_char_rep", "code_type_rep", "code_mask_rep", "code_len")}
    inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}
    inputs["src_map"] = make_src_map(batch["src_map"]).to(DEVICE)
    inputs["alignment"] = None
    inputs["blank"], inputs["fill"] = collapse_copy_scores(model.tgt_dict, batch["src_vocab"])
    return inputs

@torch.no_grad()
def predict(rows, model, beam=False):
    model.network.eval()
    translator = Translator(model, DEVICE == "cuda", beam_size=4, n_best=1,
        max_length=args.max_tgt_len, global_scorer=GNMTGlobalScorer(0, 0, "none", "none"),
        copy_attn=True, block_ngram_repeat=3, replace_unk=True) if beam else None
    builder = TranslationBuilder(model.tgt_dict, n_best=1, replace_unk=True)
    records = []
    for batch in loader(rows, model):
        assert batch["summ_word_rep"] is None and batch["tgt_seq"] is None, "Gold answer leaked into generation"
        if beam:
            result = translator.translate_batch(beam_inputs(batch, model))
            translations = builder.from_batch(result, batch["code_tokens"], [[]] * batch["batch_size"], batch["src_vocab"])
            predictions = [" ".join(t.pred_sents[0]) for t in translations]
        else:
            predictions, _, _ = model.predict(batch, replace_unk=True)
        records.extend({"id": i, "prediction": p} for i, p in zip(batch["ids"], predictions))
    assert [r["id"] for r in records] == [r["id"] for r in rows]
    return records

def load_metric(name, relative):
    spec = importlib.util.spec_from_file_location(name, ROOT / "source/codeBERT/code/eval" / relative)
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return module

bleu_metric = load_metric("codeqa_google_bleu", "bleu/google_bleu.py")
rouge_metric = load_metric("codeqa_rouge", "rouge/rouge.py")
def metrics(records, rows, extra=False):
    assert len(records) == len(rows) and [x["id"] for x in records] == [x["id"] for x in rows]
    hypotheses = {r["id"]: [p["prediction"]] for p, r in zip(records, rows)}
    references = {r["id"]: [r["answer"]] for r in rows}
    scores, yn, non = [], [], []
    for p, row in zip(records, rows):
        pred, gold = p["prediction"].lower().split(), row["answer"].lower().split()
        em = pred == gold
        common = sum((Counter(pred) & Counter(gold)).values())
        f1 = 2 * common / (len(pred) + len(gold)) if pred or gold else 1.0
        scores.append((em, f1))
        (yn if row["answer"].strip().lower() in ("yes", "no") else non).append(em)
    corpus, sentence, _ = bleu_metric.corpus_bleu(hypotheses, references)
    result = {"n": len(rows), "EM": 100 * sum(x[0] for x in scores) / len(rows),
        "F1": 100 * sum(x[1] for x in scores) / len(rows), "BLEU": 100 * sentence,
        "corpus_BLEU_diagnostic": 100 * corpus,
        "ROUGE_L": float(100 * rouge_metric.Rouge().compute_score(references, hypotheses)[0]),
        "yes_no_correct": sum(yn), "yes_no_n": len(yn), "non_yes_no_correct": sum(non), "non_yes_no_n": len(non)}
    if extra:
        meteor = None
        try:
            meteor = load_metric("codeqa_meteor", "meteor/meteor.py").Meteor()
            result["METEOR"] = 100 * meteor.compute_score(references, hypotheses)[0]
        except (OSError, RuntimeError, BrokenPipeError) as error:
            result["METEOR"] = None; result["METEOR_error"] = str(error)
        finally:
            if meteor is not None: meteor.close()
    return {key: value.item() if isinstance(value, np.generic) else value for key, value in result.items()}

# Self-check copy OOV + gradient: output dinamis harus dapat memuat identifier baru.
toy_dict = Vocabulary(); toy_dict.add("known")
toy = CopyGenerator(4, toy_dict, torch.nn.Linear(4, len(toy_dict))).to(DEVICE)
with torch.no_grad(): toy.linear_copy.weight.zero_(); toy.linear_copy.bias.fill_(20)
hidden = torch.zeros(1, 1, 4, device=DEVICE, requires_grad=True)
attention = torch.tensor([[[0., 1.]]], device=DEVICE)
source_map = torch.tensor([[[1., 0., 0.], [0., 0., 1.]]], device=DEVICE)
probabilities = toy(hidden, attention, source_map)
assert probabilities[0, 0, len(toy_dict) + 2] > .99
assert torch.allclose(probabilities.sum(-1), torch.ones(1, 1, device=DEVICE), atol=1e-6)
(-probabilities[0, 0, len(toy_dict) + 2].log()).backward()
assert hidden.grad is not None and torch.isfinite(hidden.grad).all()
# Kembali ke gate yang belum saturasi: loss OOV harus melatih gate copy.
with torch.no_grad(): toy.linear_copy.bias.zero_()
copy_scores = toy(torch.ones(1, 1, 4, device=DEVICE), attention, source_map)
copy_loss = CopyGeneratorCriterion(len(toy_dict), force_copy=False)(copy_scores,
    torch.tensor([[2]], device=DEVICE), torch.tensor([[constants.UNK]], device=DEVICE)).mean()
copy_loss.backward()
assert toy.linear_copy.bias.grad is not None and toy.linear_copy.bias.grad.abs().sum() > 0
# Bila token sama tersedia pada vocab global dan dynamic, probabilitas harus digabung.
dynamic = Vocabulary(no_special_token=True); dynamic.add("known"); dynamic.add("rare_identifier")
blank, fill = collapse_copy_scores(toy_dict, [dynamic])
assert len(toy_dict) + dynamic["known"] in blank[0]
assert len(toy_dict) + dynamic["rare_identifier"] not in blank[0]
del toy, hidden, probabilities, copy_scores, copy_loss
print("Copy dynamic OOV / normalization / gradient: PASS")

## 3. Benchmark lokal (bobot benchmark dibuang)

Mengukur beberapa batch nyata dengan vocabulary dan arsitektur penuh. Benchmark bukan prediksi waktu pasti 200 epoch: panjang urutan dan evaluasi dev ikut memengaruhi durasi. Peak tensor/reserved adalah statistik allocator PyTorch, bukan seluruh penggunaan GPU.

Jika batch 32 OOM, ubah `BATCH_SIZE` di konfigurasi lalu jalankan ulang dari awal. Notebook tidak diam-diam mengecilkan arsitektur atau batch. Perubahan batch tercatat sebagai penyimpangan konfigurasi.

In [ ]:
if not (OUT / "last.pt").exists():
    probe_model = make_model()
    probe_batches = list(loader(train_rows[:BATCH_SIZE * 3], probe_model, training=True))
    for _ in range(2): probe_model.update(probe_batches[0])
    sync()
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for batch in probe_batches: probe_model.update(batch)
    sync()
    seconds_per_batch = (time.perf_counter() - started) / len(probe_batches)
    benchmark = {"batch_size": BATCH_SIZE, "parameters": sum(p.numel() for p in probe_model.network.parameters()),
        "seconds_per_batch": seconds_per_batch,
        "estimated_full_train_epoch_seconds_excluding_dev": math.ceil(len(full_train) / BATCH_SIZE) * seconds_per_batch,
        "peak_tensor_mb": torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else None,
        "peak_reserved_mb": torch.cuda.max_memory_reserved() / 2**20 if DEVICE == "cuda" else None}
    write_json(OUT / "benchmark.json", benchmark)
    del probe_model, probe_batches, batch
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
else:
    benchmark = json.loads((OUT / "benchmark.json").read_text())
print(json.dumps(benchmark, indent=2))

## 4. Training + checkpoint/resume

Mode smoke memakai 128 train, 32 dev, maksimal 2 epoch. Mode medium memakai 10.000 train, 500 dev, maksimal 20 epoch dan patience 4. Mode paper memakai semua data dan batas epoch/patience paper. Validasi memakai generasi greedy seperti loop training source rujukan; checkpoint dipilih dari BLEU dev. Test memakai beam 4.

Checkpoint `last.pt` menyimpan optimizer, posisi batch berikutnya, dan RNG; `best.pt` menyimpan bobot terbaik. Batas waktu dihitung per sesi eksekusi. Jika sesi berhenti, jalankan ulang notebook dengan konfigurasi sama dan `RESUME=True`. Checkpoint milik notebook ini hanya berisi tensor dan tipe Python standar.

In [ ]:
model = make_model()
history, epoch, next_batch, no_improvement = [], 1, 0, 0
best_bleu, best_epoch, train_seconds_total = -1.0, 0, 0.0
stop_reason = "not_started"

def save_checkpoint(path, next_epoch, batch_offset):
    payload = {"run_hash": run_hash, "model": model.network.state_dict(),
        "optimizer": model.optimizer.state_dict(), "updates": model.updates,
        "epoch": next_epoch, "next_batch": batch_offset, "history": history,
        "best_bleu": best_bleu, "best_epoch": best_epoch, "no_improvement": no_improvement,
        "train_seconds_total": train_seconds_total, "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else []}
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary); temporary.replace(path)

if (OUT / "last.pt").exists():
    assert RESUME, "Hasil run sudah ada. Gunakan RESUME=True atau RUN_TAG baru."
    saved = torch.load(OUT / "last.pt", map_location="cpu", weights_only=True)
    assert saved["run_hash"] == run_hash
    model.network.load_state_dict(saved["model"])
    model.init_optimizer(saved["optimizer"], use_gpu=DEVICE == "cuda")
    model.updates = saved["updates"]
    epoch, next_batch, history = saved["epoch"], saved["next_batch"], saved["history"]
    best_bleu, best_epoch, no_improvement = saved["best_bleu"], saved["best_epoch"], saved["no_improvement"]
    train_seconds_total = saved["train_seconds_total"]
    torch.set_rng_state(saved["torch_rng"])
    if DEVICE == "cuda" and saved["cuda_rng"]: torch.cuda.set_rng_state_all(saved["cuda_rng"])
    del saved
    print("Resume epoch/batch:", epoch, next_batch)

# Checkpoint awal menjamin selalu ada titik aman sebelum optimizer mulai diperbarui.
if not (OUT / "last.pt").exists(): save_checkpoint(OUT / "last.pt", epoch, next_batch)
session_start = time.perf_counter()
batches_per_epoch = math.ceil(len(train_rows) / BATCH_SIZE)
try:
    while epoch <= MAX_EPOCHS and no_improvement < PATIENCE:
        # Upstream decays even at epoch 1; assignment also makes resume deterministic.
        model.optimizer.param_groups[0]["lr"] = args.learning_rate * args.lr_decay ** epoch
        batch_losses = []
        for batch_index, batch in enumerate(loader(train_rows, model, training=True, epoch=epoch, start_batch=next_batch), start=next_batch):
            tick = time.perf_counter()
            update = model.update(batch)
            sync(); train_seconds_total += time.perf_counter() - tick
            assert math.isfinite(update["ml_loss"]), "Non-finite training loss"
            batch_losses.append(update["ml_loss"])
            next_batch = batch_index + 1
            if next_batch % 100 == 0:
                save_checkpoint(OUT / "last.pt", epoch, next_batch)
                print(f"Epoch {epoch}: batch {next_batch}/{batches_per_epoch}; loss {update['ml_loss']:.3f}")
            if time.perf_counter() - session_start >= MAX_SESSION_SECONDS:
                stop_reason = "time_budget"; break
        if stop_reason == "time_budget": break
        dev_predictions = predict(dev_rows, model, beam=False)
        dev_scores = metrics(dev_predictions, dev_rows)
        record = {"epoch": epoch, "train_mean_sequence_loss_this_session": sum(batch_losses)/len(batch_losses) if batch_losses else None,
            "dev": dev_scores, "lr": model.optimizer.param_groups[0]["lr"]}
        history.append(record)
        if dev_scores["BLEU"] > best_bleu:
            best_bleu, best_epoch, no_improvement = dev_scores["BLEU"], epoch, 0
            temporary = OUT / "best.tmp"
            torch.save({"run_hash": run_hash, "epoch": epoch, "state_dict": model.network.state_dict()}, temporary)
            temporary.replace(OUT / "best.pt")
        else: no_improvement += 1
        print(f"Epoch {epoch}: dev BLEU={dev_scores['BLEU']:.2f}; EM={dev_scores['EM']:.2f}; F1={dev_scores['F1']:.2f}; best={best_epoch}")
        epoch += 1; next_batch = 0
        write_json(OUT / "history.json", history)
        save_checkpoint(OUT / "last.pt", epoch, next_batch)
        if time.perf_counter() - session_start >= MAX_SESSION_SECONDS:
            stop_reason = "time_budget"; break
    if stop_reason != "time_budget":
        stop_reason = "early_stopping" if no_improvement >= PATIENCE else "max_epochs"
except KeyboardInterrupt:
    stop_reason = "interrupted"
    print("Interupsi: checkpoint utuh terakhir dipertahankan; maksimal 99 batch akan diulang.")
except Exception:
    stop_reason = "error"
    raise
finally:
    # Jangan menyimpan optimizer yang mungkin baru ter-update sebagian saat interupsi/error.
    if stop_reason not in ("interrupted", "error"):
        save_checkpoint(OUT / "last.pt", epoch, next_batch)
    write_json(OUT / "history.json", history)

training_complete = stop_reason in ("max_epochs", "early_stopping")
status = {"profile": PROFILE, "stop_reason": stop_reason, "training_complete_for_profile": training_complete,
    "full_paper_training_completed": PROFILE == "paper" and training_complete,
    "epoch_next": epoch, "next_batch": next_batch, "best_epoch": best_epoch,
    "best_dev_BLEU": best_bleu, "training_seconds_total_excluding_dev_io": train_seconds_total,
    "session_wall_seconds": time.perf_counter() - session_start}
write_json(OUT / "status.json", status)
print(json.dumps(status, indent=2))

## 5. Checkpoint terbaik, beam 4, dan evaluasi test

Test dibuka setelah run selesai untuk profilnya. Smoke memakai 16 test acak, medium memakai 500 test acak, dan paper memakai seluruh 7.000 test. Hasil smoke/medium bersifat eksploratif dan tidak sebanding langsung dengan skor penuh paper. Jika run terhenti karena batas waktu, evaluasi test dilewati kecuali `EVALUATE_PARTIAL=True` dipilih secara eksplisit; jangan memakai skor test untuk memilih konfigurasi.

BLEU adalah rata-rata sentence BLEU smoothed pada jalur `eval_accuracies`; corpus BLEU diberi label diagnostik. EM/F1 mengikuti lowercase + whitespace normalization. ROUGE-L dan METEOR memakai modul source CodeQA. Tidak menggunakan tokenizer regex Mini V3 untuk skor utama.

In [ ]:
if not (OUT / "best.pt").exists():
    print("Belum ada best checkpoint. Resume training sampai satu evaluasi dev selesai.")
elif not training_complete and not EVALUATE_PARTIAL:
    print("Training masih parsial; test belum dibuka. Jalankan ulang untuk resume.")
else:
    saved_best = torch.load(OUT / "best.pt", map_location="cpu", weights_only=True)
    assert saved_best["run_hash"] == run_hash
    model.network.load_state_dict(saved_best["state_dict"])
    del saved_best
    # Test reload dan decoder beam upstream pada input identik.
    probe = dev_rows[:2]
    first = predict(probe, model, beam=True)
    saved_best = torch.load(OUT / "best.pt", map_location="cpu", weights_only=True)
    model.network.load_state_dict(saved_best["state_dict"]); del saved_best
    second = predict(probe, model, beam=True)
    assert first == second, "Reload checkpoint mengubah hasil beam decoding"
    print("Reload + beam4 consistency: PASS")
    test_full = load_rows("test")
    test_count = {"smoke": 16, "medium": 500, "paper": len(test_full)}[PROFILE]
    test_rows = test_full if PROFILE == "paper" else choose(test_full, test_count, SEED + 2)
    sync(); started = time.perf_counter()
    test_records = predict(test_rows, model, beam=True)
    sync(); test_seconds = time.perf_counter() - started
    result = metrics(test_records, test_rows, extra=True)
    result.update(status)
    result.update({"evaluation_scope": {"smoke": "smoke_subset_only", "medium": "medium_subset_only", "paper": "full_test"}[PROFILE],
        "test_inference_seconds": test_seconds, "best_checkpoint_sha256": sha256(OUT / "best.pt"),
        "test_data_hashes": {k: v for k, v in DATA_HASHES.items() if k.startswith("test.")},
        "paper_python_seq2seq_reference": {"BLEU": 30.09, "ROUGE_L": 21.69, "METEOR": 6.38, "EM": 2.84, "F1": 23.27}})
    for i, (record, row) in enumerate(zip(test_records, test_rows)):
        assert record["id"] == row["id"]
        test_records[i] = {**row, **record}
    temporary = OUT / "test_predictions.tmp"
    with temporary.open("w", encoding="utf-8") as stream:
        for record in test_records: stream.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(OUT / "test_predictions.jsonl")
    write_json(OUT / "results.json", result)
    print(json.dumps(result, indent=2))
    print("\nContoh pertama tanpa seleksi:")
    for record in test_records[:5]:
        print("Q:", record["question"], "\nGold:", record["answer"], "\nPred:", record["prediction"], "\n")

## 6. Coba kode dan pertanyaan sendiri

Hanya memakai checkpoint terbaik yang aktif. Jawaban referensi tidak diperlukan. Hasil smoke belum menunjukkan kemampuan QA yang memadai.

In [ ]:
def answer(code_text, question_text):
    if not (OUT / "best.pt").exists():
        raise RuntimeError("Latih minimal satu epoch dan muat best checkpoint dahulu")
    row = {"id": 0, "code": code_text, "question": question_text}
    return predict([row], model, beam=True)[0]["prediction"]

# Setelah training paper selesai, ganti isi dua variabel ini.
code_text = "def is_adult(age): return age >= 18"
question_text = "When does the function return true ?"
if (OUT / "best.pt").exists() and training_complete:
    print("Question:", question_text)
    print("Answer:", answer(code_text, question_text))

## Batas interpretasi

- Notebook merekonstruksi baseline Seq2seq CodeQA memakai source model terkait, bukan source Seq2seq CodeQA yang tidak dirilis di repo tersebut.
- LSTM, embedding/vocabulary, batas panjang, LR decay, greedy dev, dan beberapa detail decoding mengikuti source rujukan saat paper tidak merinci semuanya.
- Vocab train-only berbeda dari loader upstream yang dapat membangun dictionary dari file train+dev. Tidak ada kebocoran dev/test untuk vocab notebook ini.
- Smoke dan medium mengurangi data/epoch, **bukan dimensi model**. Skornya tidak dapat dibandingkan langsung dengan Table 8.
- Batas dua jam, batch yang diubah pengguna, atau run belum konvergen harus dilaporkan sebagai perbedaan eksperimen.
- Python dan PyTorch modern berbeda dari runtime upstream. Patch kompatibilitas disimpan beserta hash original dan diff; bobot lama project tidak dipakai.
- Ada potensi overlap kode antar split resmi karena split dilakukan pada pasangan QA. Notebook mempertahankan split penulis, tidak mengklaim evaluasi lintas-repositori.
- Hasil negatif tetap sah. Jangan memilih seed/konfigurasi berdasarkan test atau menyebut skor smoke/medium sebagai reproduksi penuh.
- Saat kode notebook berubah, hash run juga berubah. Checkpoint lama tetap tersimpan dalam direktori run lama, tetapi tidak otomatis di-resume oleh versi kode baru.